In [13]:
import pandas as pd
import numpy as np

FOUR_PT_LINE = 30                  
N_SIMS = 10_000                    
SHIFTS = [0.0, 0.10, 0.20, 0.30] 
PENALTY = 0.02
rng = np.random.default_rng(42)
NEAR_FEET = [FOUR_PT_LINE - 2, FOUR_PT_LINE - 1]
LANDING_FEET = [FOUR_PT_LINE, FOUR_PT_LINE + 1]
MIN_ATTEMPTS = 20

shots = pd.read_csv("dataset2_top50_3pm_by_distance.csv")
shots["Distance"] = pd.to_numeric(shots["Shot Distance (ft)"].replace("40+", "40"))
summary = pd.read_csv("dataset2_top50_3pm_summary.csv")

keys = ["Season", "Player ID", "Player", "Team"]

def totals(mask, name):
    return (shots[mask].groupby(keys)[["3PM", "3PA"]].sum().rename(columns={"3PM": f"{name}_M", "3PA": f"{name}_A"}))

deep = totals(shots["Distance"] >= FOUR_PT_LINE, "deep")          
near = totals(shots["Distance"].isin(NEAR_FEET), "near")          
landing = totals(shots["Distance"].isin(LANDING_FEET), "land") 

players = (summary[keys + ["GP"]].merge(deep, on=keys, how="left").merge(near, on=keys, how="left").merge(landing, on=keys, how="left").fillna(0))

def make_pct(makes, attempts):
    league = makes.sum() / attempts.sum()
    return np.where(attempts >= MIN_ATTEMPTS, makes / attempts.clip(lower=1), league)

players["p_deep"] = make_pct(players["deep_M"], players["deep_A"])
players["p_near"] = make_pct(players["near_M"], players["near_A"])
players["p_moved"] = make_pct(players["land_M"], players["land_A"])

def simulate(row, shift):
    deep_attempts = rng.poisson(row["deep_A"], N_SIMS)            
    moved_attempts = rng.poisson(shift * row["near_A"], N_SIMS)   

    deep_makes = rng.binomial(deep_attempts, row["p_deep"])       
    moved_makes = rng.binomial(moved_attempts, row["p_moved"])
    moved_makes_as_3 = rng.binomial(moved_attempts, row["p_near"])

    extra_points = deep_makes * 1 + moved_makes * 4 - moved_makes_as_3 * 3
    return extra_points / row["GP"]

gain_rows = []
for shift in SHIFTS:
    for _, row in players.iterrows():
        s = simulate(row, shift)
        gain_rows.append({"Season": row["Season"], "Player": row["Player"], "Team": row["Team"], "Shift": shift, "Extra PPG (avg)": s.mean(), "Low (5%)": np.percentile(s, 5), "High (95%)": np.percentile(s, 95)})
gains = pd.DataFrame(gain_rows)
gains.to_csv("part1_gains.csv", index=False)

pd.set_option("display.width", 140)
for season in sorted(gains["Season"].unique()):
    print(f"\n===== {season}: top 10 gainers, no behavior change (shift 0%) =====")
    top = (gains[(gains["Season"] == season) & (gains["Shift"] == 0)]
           .sort_values("Extra PPG (avg)", ascending=False).head(10))
    print(top[["Player", "Team", "Extra PPG (avg)", "Low (5%)", "High (95%)"]]
          .round(2).to_string(index=False))

print("\n===== Average extra PPG for the top-50 3PT shooters, by shift =====")
print(gains.pivot_table(index="Season", columns="Shift",
                        values="Extra PPG (avg)", aggfunc="mean").round(2))

print("\nSaved: part1_gains.csv (extra points for every player, season and shift)")





===== 2021-22: top 10 gainers, no behavior change (shift 0%) =====
          Player Team  Extra PPG (avg)  Low (5%)  High (95%)
      Trae Young  ATL             0.59      0.45        0.74
  Darius Garland  CLE             0.40      0.28        0.53
   Stephen Curry  GSW             0.33      0.22        0.45
     Luka Dončić  DAL             0.20      0.11        0.29
    LeBron James  LAL             0.18      0.09        0.29
     LaMelo Ball  CHA             0.17      0.09        0.25
Donovan Mitchell  UTA             0.15      0.07        0.22
   Fred VanVleet  TOR             0.14      0.06        0.22
      Kevin Love  CLE             0.12      0.05        0.19
     Buddy Hield  IND             0.10      0.05        0.16

===== 2022-23: top 10 gainers, no behavior change (shift 0%) =====
           Player Team  Extra PPG (avg)  Low (5%)  High (95%)
   Damian Lillard  POR             0.65      0.48        0.83
       Trae Young  ATL             0.36      0.25        0.48
     Jo